# SDOA LoRA Fine-Tune — Google Colab

**What this notebook does:**
1. Installs Unsloth (2× faster QLoRA, optimised for T4/A100)
2. Loads `Qwen2.5-7B-Instruct` in 4-bit
3. Fine-tunes on your SDOA doctrine dataset (104 instruction-response pairs)
4. Exports the merged model as `sdoa-lora.Q4_K_M.gguf`
5. Downloads it so you can drop it into your local Ollama setup

**Runtime required:** GPU — go to `Runtime → Change runtime type → T4 GPU` (free tier works)

**Estimated time:** ~45–60 min on T4, ~20 min on A100

---

In [ ]:
# Cell 0 — Fix torchaudio/torch version conflict
# torchaudio conflicts with PyTorch 2.7+ in Colab and is not needed for text fine-tuning.
# Run this cell first, then Runtime > Restart session, then continue from Cell 1.
!pip uninstall torchaudio -y -q
print("Done. Now go to Runtime > Restart session, then run Cell 1 onwards.")

## Cell 1 — Install Unsloth

In [ ]:
%%capture
# Standard HuggingFace QLoRA — no Unsloth dependency, full dtype control.
!pip install torch transformers datasets peft trl bitsandbytes accelerate
print("Done.")

## Cell 2 — Upload your SDOA dataset

Run this cell, then click **Choose Files** and upload `datasets/sdoa_lora_dataset.jsonl` from your local project.

_(Alternative: mount Google Drive and copy the file there first.)_

In [ ]:
from google.colab import files
import json, pathlib

uploaded = files.upload()   # upload sdoa_lora_dataset.jsonl

dataset_path = list(uploaded.keys())[0]
records = [json.loads(l) for l in pathlib.Path(dataset_path).read_text().splitlines() if l.strip()]
print(f"Loaded {len(records)} training pairs from {dataset_path}")

## Cell 3 — Load Qwen2.5-7B-Instruct in 4-bit

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

BASE_MODEL     = "Qwen/Qwen2.5-3B-Instruct"   # ~6GB download, ~2GB in 4-bit VRAM
MAX_SEQ_LENGTH = 2048

bnb_config = BitsAndBytesConfig(
    load_in_4bit              = True,
    bnb_4bit_quant_type       = "nf4",
    bnb_4bit_compute_dtype    = torch.float16,
    bnb_4bit_use_double_quant = True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config = bnb_config,
    device_map          = "auto",
    trust_remote_code   = True,
)
model.config.use_cache = False
print("Model loaded.")

## Cell 4 — Apply LoRA adapter (rank 16)

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r              = 16,
    lora_alpha     = 32,
    lora_dropout   = 0.05,
    bias           = "none",
    task_type      = "CAUSAL_LM",
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## Cell 5 — Format dataset into Qwen2.5 chat template

In [ ]:
from datasets import Dataset

def to_text(record):
    """Convert ShareGPT record to Qwen2.5 <|im_start|> chat format."""
    parts = []
    for turn in record.get("conversations", []):
        role  = turn["from"]
        value = turn["value"]
        if role == "system":
            parts.append(f"<|im_start|>system\n{value}<|im_end|>")
        elif role == "human":
            parts.append(f"<|im_start|>user\n{value}<|im_end|>")
        elif role == "gpt":
            parts.append(f"<|im_start|>assistant\n{value}<|im_end|>")
    return {"text": "\n".join(parts)}

dataset = Dataset.from_list([to_text(r) for r in records])
print(f"Dataset ready: {len(dataset)} samples")
print("Sample preview:")
print(dataset[0]["text"][:400])

## Cell 6 — Fine-tune (QLoRA, 3 epochs, ~45 min on T4)

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model              = model,
    tokenizer          = tokenizer,
    train_dataset      = dataset,
    dataset_text_field = "text",
    max_seq_length     = MAX_SEQ_LENGTH,
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        num_train_epochs            = 3,
        warmup_steps                = 10,
        learning_rate               = 2e-4,
        lr_scheduler_type           = "cosine",
        fp16                        = True,
        bf16                        = False,
        logging_steps               = 10,
        optim                       = "paged_adamw_8bit",
        weight_decay                = 0.01,
        packing                     = True,
        output_dir                  = "/content/sdoa-checkpoints",
        report_to                   = "none",
    ),
)

trainer_stats = trainer.train()
print(f"Training complete. Runtime: {trainer_stats.metrics['train_runtime']:.0f}s")

# Save adapter immediately — trainer.model is the PEFT wrapper (~50MB, not 2GB)
trainer.model.save_pretrained("/content/sdoa-adapter")
tokenizer.save_pretrained("/content/sdoa-adapter")
print("Adapter saved.")

## Cell 7 — Export to GGUF (Q4_K_M)

This merges the LoRA weights into the base model and quantizes to GGUF format.
The output file is what you drop into your local Ollama setup.

In [ ]:
import os, glob, gc, torch
from peft import PeftModel

print("Loading adapter onto base model...")
peft_model = PeftModel.from_pretrained(model, "/content/sdoa-adapter")

print("Merging...")
merged = peft_model.merge_and_unload()
merged.save_pretrained("/content/sdoa-merged", safe_serialization=True)
tokenizer.save_pretrained("/content/sdoa-merged")
print("Merged. Converting to GGUF...")

del peft_model, merged
gc.collect()
torch.cuda.empty_cache()

if not os.path.exists("/content/llama.cpp"):
    !git clone --depth 1 https://github.com/ggerganov/llama.cpp /content/llama.cpp -q
!pip install -r /content/llama.cpp/requirements.txt -q
!python /content/llama.cpp/convert_hf_to_gguf.py /content/sdoa-merged \
    --outfile /content/sdoa-lora.gguf --outtype q4_k_m

print(f"GGUF: {glob.glob('/content/*.gguf')}")

## Cell 8 — Download the GGUF

This downloads `sdoa-lora.Q4_K_M.gguf` to your browser.

**After downloading:**
1. Place the file at `models/sdoa-lora.gguf` in your local project
2. In `ollama/Modelfile.sdoa`, make sure the `ADAPTER` line is **uncommented**
3. Run: `ollama create sdoa-qwen -f ollama/Modelfile.sdoa`
4. Run: `node scripts/sdoa_model_validate.js`

In [ ]:
from google.colab import files
import glob, os

gguf_files = glob.glob("/content/sdoa-lora*.gguf")
if not gguf_files:
    print("ERROR: No GGUF file found. Did Cell 7 complete successfully?")
else:
    gguf_path = gguf_files[0]
    size_gb   = os.path.getsize(gguf_path) / 1e9
    print(f"Downloading {gguf_path} ({size_gb:.1f} GB)...")
    files.download(gguf_path)

## (Optional) Cell 9 — Save to Google Drive instead of downloading

Use this if the file is too large to download directly (~4.5 GB).
Mount Drive first, then copy the file there.

In [ ]:
# from google.colab import drive
# drive.mount("/content/drive")
# import shutil, glob
# gguf = glob.glob("/content/sdoa-lora*.gguf")[0]
# dest = "/content/drive/MyDrive/sdoa-lora.gguf"
# shutil.copy(gguf, dest)
# print(f"Saved to Google Drive: {dest}")

---

## After the download — local steps

```bash
# 1. Place the GGUF in your project
mkdir -p models
mv ~/Downloads/sdoa-lora.Q4_K_M.gguf models/sdoa-lora.gguf

# 2. Register in Ollama (uses Modelfile.sdoa which has ADAPTER sdoa-lora.gguf)
#    NOTE: Ollama resolves the ADAPTER path relative to the Modelfile directory
#    so either run from ollama/ or pass the absolute path
cp models/sdoa-lora.gguf ollama/sdoa-lora.gguf
ollama create sdoa-qwen -f ollama/Modelfile.sdoa

# 3. Run the certification suite
node scripts/sdoa_model_validate.js --verbose

# Target: >= 92% to proceed to Track 2 (Sleeve integration)
```